In [18]:
import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go

import numpy as np
from scipy.stats import gaussian_kde
import plotly.graph_objects as go

In [ ]:
## === Get DataFrame from database of a specific group ===
gname = 'Grupo Deus é Amor'  # target group name

conn = sqlite3.connect("dataset/railway.sqlite")
# cursor = conn.cursor()

query = "\
    SELECT e.id AS event_id, e.name AS event_name, e.start_date_time AS event_time, \
        p.id AS participant_id, p.full_name AS participant_name, c.timestamp AS checkin_time, \
        p.birth_date AS participant_birth, p.gender AS participant_gender \
    FROM check_ins AS c \
    JOIN events AS e ON c.event_id = e.id \
    JOIN participants AS p ON c.participant_id = p.id \
    WHERE e.group_id = (SELECT id FROM groups WHERE name = ?)"
# cursor.execute(query, (gname,))
# results = cursor.fetchall()
df = pd.read_sql(query, conn, params=(gname,))

In [3]:
### === Data Cleaning and Preprocessing ===

# # Check for missing values
# df.isnull().sum()
# df.isna().sum()

# Convert date columns to datetime format
for col in ["event_time", "checkin_time", "participant_birth"]:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    
# Add age column
df["participant_age"] = (pd.to_datetime("today") - df["participant_birth"]).dt.days // 365

# De-identify participants by only keeping first names
df["participant_name"] = df["participant_name"].str.split().str[0]

In [4]:
# # abnormal age value check
# abnormal_age_ids = df[df["participant_age"] <= 7]["participant_id"].drop_duplicates().tolist()

# # Get contact info for abnormal age participants
# placeholders = ",".join("?" * len(abnormal_age_ids))
# query = f"SELECT id, full_name, email, phone, birth_date FROM participants WHERE id IN ({placeholders})"

# contact_info = pd.read_sql(query, conn, params=abnormal_age_ids)
# contact_info.to_csv("dataset/abnormal_age_participants.csv", index=False)

conn.close()

In [6]:
df.dtypes

event_id                      object
event_name                    object
event_time            datetime64[ns]
participant_id                object
participant_name              object
checkin_time          datetime64[ns]
participant_birth     datetime64[ns]
participant_gender            object
participant_age                int64
dtype: object

## Group Participants Analysis

In [5]:
# all registered participants in the group
participants = df[["participant_id", "participant_name", "participant_gender", "participant_age"]].drop_duplicates().reset_index(drop=True)

# filtering age out of bound participants
participants = participants[(participants["participant_age"] > 7) & (participants["participant_age"] < 30)].reset_index(drop=True)

# add attendance count
attendance = df.groupby("participant_id").size().reset_index(name="attendance_count")
participants = participants.merge(attendance, on="participant_id", how="left")

# rename columns for better readability
participants.columns = ["id", "name", "gender", "age", "attendance_count"]
participants.head()

,id,name,gender,age,attendance_count
0,59fa3bcd-4903-4349-ac44-4af3352efc93,Felipe,MALE,16,2
1,346c4a15-bf18-4917-9877-48df46d84fdc,Felipe,MALE,13,1
2,13762581-ab39-4b01-ab0b-274ac8c6bceb,Antônio,MALE,17,2
3,4226528c-c1c4-4ed5-ad7f-539378323e31,Otávio,MALE,16,2
4,4cd326fe-1c19-414c-9740-094e5030f90c,Francisco,MALE,16,2


### Gender & Age Analysis

In [6]:
# Gender distribution
gender = participants["gender"]
gender_info = pd.DataFrame({'count': gender.value_counts(), 
                            'percentage': (gender.value_counts(normalize=True) * 100).round(2)})
gender_info.reset_index(inplace=True)
gender_info

,gender,count,percentage
0,FEMALE,312,55.61
1,MALE,249,44.39


In [40]:
# Age distribution (overall)
def get_age_info(age_data, freq=False):
    print(f"Average age: {age_data.mean():.1f} \nMedian age: {age_data.median()} \nAge range: {age_data.min()} - {age_data.max()}")
    if freq:
        print("\nMost frequent ages:")
        values = age_data.value_counts().reset_index()
        for row in values.head(3).itertuples():
            print(f"Age {row[1]}: {row[2]} participants")

overall_age = participants['age']
get_age_info(overall_age)

Average age: 14.7 
Median age: 14.0 
Age range: 11 - 25


In [ ]:
age_df = overall_age.value_counts().reset_index()
# (age_count/age_count.sum()).cumsum()
age_df['percentage'] = age_df['count']/age_df['count'].sum()
age_df['cumulative_percentage'] = age_df['percentage'].cumsum()

# pareto = go.Figure([
#     go.Bar(x=age_df['age'], y=age_df['count'], name='Age Count'),
#     go.Scatter(x=age_df['age'], y=age_df['cumulative_percentage'], 
#                mode='lines+markers', yaxis='y2', name='Cumulative %')]) 
# pareto = pareto.update_layout(yaxis2=dict(overlaying='y', side='right', tickformat='.0%', range=[0, 1]))
# pareto.add_hline(y=0.8, yref='y2', line_dash="dash")

age_df.head()

,age,count,percentage,cumulative_percentage
0,14,137,0.244207,0.244207
1,15,129,0.229947,0.474153
2,13,107,0.190731,0.664884
3,16,86,0.153298,0.818182
4,12,35,0.062389,0.880570


More than 80% of all participants are of age 13 - 16

In [167]:
hist_age = px.histogram(participants, x="age", 
             title="Age Distribution of Participants")
hist_age.update_layout(bargap=0.2, 
                       width=600, height=400)
hist_age.update_xaxes(dtick=1)
hist_age.update_traces(marker=dict(color="#dd9add", line=dict(color='black', width=1)))

In [41]:
age_gender = participants.groupby(["gender", "age"], as_index=False).size()
age_gender.sort_values(by=['size'], inplace=True, ascending=False)
age_gender['pct'] = age_gender['size'] / age_gender['size'].sum()
age_gender['cumulative_pct'] = age_gender['pct'].cumsum()

age_gender.head()

,gender,age,size,pct,cumulative_pct
2,FEMALE,13,80,0.142602,0.142602
3,FEMALE,14,76,0.135472,0.278075
11,MALE,15,69,0.122995,0.401070
10,MALE,14,61,0.108734,0.509804
4,FEMALE,15,60,0.106952,0.616756


In [42]:
female_age = participants[participants["gender"] == "FEMALE"]["age"]
male_age = participants[participants["gender"] == "MALE"]["age"]
print("Female Participants Age Info:")
get_age_info(female_age, freq=True)
print("\nMale Participants Age Info:")
get_age_info(male_age, freq=True)


Female Participants Age Info:
Average age: 14.3 
Median age: 14.0 
Age range: 11 - 18

Most frequent ages:
Age 13: 80 participants
Age 14: 76 participants
Age 15: 60 participants

Male Participants Age Info:
Average age: 15.1 
Median age: 15.0 
Age range: 12 - 25

Most frequent ages:
Age 15: 69 participants
Age 14: 61 participants
Age 16: 42 participants


In [161]:
# histogram: age distribution by gender 
hist_age_gender = px.histogram(participants, x='age',
             color='gender',
             barmode='group', # 'group' - side by side, 'stack' - stacked bars
             # histnorm='probability density',
             title='Age Distribution By Gender',
             color_discrete_map={'MALE': '#87ceeb', 'FEMALE': '#ffb6c1'})
hist_age_gender.update_layout(bargap=0.3, width=600, height=400)
hist_age_gender.update_traces(marker=dict(line=dict(color='black', width=1)))
hist_age_gender.update_xaxes(dtick=1)

# # Add KDE automatically per gender
# for gender, group in participants.groupby('gender'):
#     ages = group['age'].dropna()
#     kde = gaussian_kde(ages)
#     x_range = np.linspace(ages.min(), ages.max(), 100)
#     hist.add_trace(go.Scatter(x=x_range, y=kde(x_range), mode='lines', name=f'{gender} curve'))


In [168]:
# px.violin(participants, x="gender", y="age", 
#           color="gender", box=True,
#           title="Age Distribution by Gender")

Attendance Analysis

In [ ]:
# Full attendance
full_att_pct = len(participants[participants["attendance_count"] == 3]) / len(participants)
print(f"Full attendance rate: {full_att_pct:.2%}")

Full attendance rate: 16.40%


In [ ]:
# gender vs attendance
att_by_gender = participants.groupby("gender")["attendance_count"]
print(att_by_gender.agg(['mean', 'std', 'var']))
# px.box(participants, x="gender", y="attendance_count", color="gender") ## not useful, two boxes are the same
# px.histogram(participants, x="attendance_count", barmode="group", color="gender", opacity=0.75) ## not useful, more female than male, comparison not meaningful

            mean       std       var
gender                              
FEMALE  1.676282  0.782248  0.611912
MALE    1.530120  0.707175  0.500097


In [59]:
# age vs attendance
att_by_age = participants.groupby("age")["attendance_count"]
# print(att_by_age.mean())
participants[['age', 'attendance_count']].corr()

,age,attendance_count
age,1.000000,-0.005329
attendance_count,-0.005329,1.000000


In [57]:
px.scatter(participants, x='age', y='attendance_count',
           color='gender',  # optional, adds gender context
           trendline='ols')             # adds a regression trendline

In [ ]:
# att_by_age.mean().sort_values(ascending=False)
# px.bar(att_by_age.mean().reset_index(), x='age', y='attendance_count')

In [48]:
att_age_gender = participants.groupby(['gender','age'])['attendance_count']
engagement = att_age_gender.agg(['mean', 'count']).reset_index()
engagement = engagement[engagement['count'] >= 5] # remove groups too small to be statistically meaningful.
engagement = engagement.sort_values(by=['mean'], ascending=False)
engagement.head(3), engagement.tail(3)

(    gender  age      mean  count
 5   FEMALE   16  1.818182     44
 13    MALE   17  1.750000     16
 4   FEMALE   15  1.733333     60,
    gender  age      mean  count
 10   MALE   14  1.442623     61
 14   MALE   18  1.375000      8
 15   MALE   19  1.000000      5)

In [54]:
fig = px.bar(engagement.sort_values('age'), 
             x='age', y='mean', color='gender',
             barmode='group', text='mean', # shows the mean value on each bar
             color_discrete_map={'MALE': '#87ceeb', 'FEMALE': '#ffb6c1'})  
fig.update_traces(texttemplate='%{text:.2f}',  # 2 decimal places
                  textposition='outside')
fig.update_xaxes(dtick=1)  # show every age on x-axis
fig.update_layout(xaxis_title='Age', yaxis_title='Average Attendance',
                  legend_title='Gender')

fig.show()

In [71]:
var = att_age_gender.agg(['mean', 'std', 'var']).reset_index()
var.sort_values(by=['std'], ascending=False)
# Higher std = more irregular attendance within that group.

,gender,age,mean,std,var
0,FEMALE,11,1.666667,1.154701,1.333333
16,MALE,20,2.000000,1.000000,1.000000
7,FEMALE,18,1.555556,0.881917,0.777778
5,FEMALE,16,1.813953,0.852331,0.726467
6,FEMALE,17,1.705882,0.848875,0.720588
2,FEMALE,13,1.705128,0.823502,0.678155
13,MALE,17,1.750000,0.774597,0.600000
4,FEMALE,15,1.706897,0.772524,0.596794
11,MALE,15,1.608696,0.771122,0.594629
14,MALE,18,1.375000,0.744024,0.553571


## Arrival rush by min

In [84]:
df.head()

,event_id,event_name,event_time,participant_id,participant_name,checkin_time,participant_birth,participant_gender,participant_age
0,171efb44-3973-4f55-bad0-8d32f53128ea,12/03,2026-03-12 18:30:00,59fa3bcd-4903-4349-ac44-4af3352efc93,Felipe,2026-03-12 21:56:04.693167,2009-11-19,MALE,16
1,171efb44-3973-4f55-bad0-8d32f53128ea,12/03,2026-03-12 18:30:00,346c4a15-bf18-4917-9877-48df46d84fdc,Felipe,2026-03-12 22:13:15.777129,2012-07-19,MALE,13
2,171efb44-3973-4f55-bad0-8d32f53128ea,12/03,2026-03-12 18:30:00,13762581-ab39-4b01-ab0b-274ac8c6bceb,Antônio,2026-03-12 22:13:40.541487,2008-09-13,MALE,17
3,171efb44-3973-4f55-bad0-8d32f53128ea,12/03,2026-03-12 18:30:00,4226528c-c1c4-4ed5-ad7f-539378323e31,Otávio,2026-03-12 22:17:05.458682,2010-03-04,MALE,16
4,171efb44-3973-4f55-bad0-8d32f53128ea,12/03,2026-03-12 18:30:00,4cd326fe-1c19-414c-9740-094e5030f90c,Francisco,2026-03-12 22:17:35.558127,2010-03-30,MALE,16


In [98]:
checkin_info = df[['event_name', 'checkin_time']].copy()
checkin_info['hour'] = checkin_info['checkin_time'].dt.hour
checkin_info['minute'] = checkin_info['checkin_time'].dt.minute
checkin_info['hr_min'] = checkin_info['checkin_time'].dt.strftime('%H:%M')
checkin_info.head()

,event_name,checkin_time,hour,minute,hr_min
0,12/03,2026-03-12 21:56:04.693167,21,56,21:56
1,12/03,2026-03-12 22:13:15.777129,22,13,22:13
2,12/03,2026-03-12 22:13:40.541487,22,13,22:13
3,12/03,2026-03-12 22:17:05.458682,22,17,22:17
4,12/03,2026-03-12 22:17:35.558127,22,17,22:17


In [ ]:
checkin_info["hr_min"].describe()

count       924
unique       54
top       22:30
freq         56
Name: hr_min, dtype: object

In [ ]:
# Check-in minute rush by event
min_rush = checkin_info.groupby(['event_name', 'hr_min'], as_index=False).size()
# min_rush = min_rush.sort_values(by=['hr_min'])
time_order = sorted(min_rush['hr_min'].unique(), key=lambda x: pd.to_datetime(x, format='%H:%M'))
min_fig = px.line(min_rush, x='hr_min', y='size', 
                  category_orders={'hr_min': time_order},  # ensure x-axis is in chronological order
                  color='event_name', markers=True,
                  title='Check-in Count by Time')
min_fig.update_xaxes(dtick=3, tickangle=30)  
min_fig.update_layout(xaxis_title='Time (Hour:Minute)', yaxis_title='Number of Check-ins')
min_fig

In [ ]:
# Overall check-in time distribution